# 07 — Content Addressing Specifies Model Distribution

**Leading specification:** identity follows content, not storage location.

Notebook 00 established the repository vocabulary. Notebook 07 develops the first mechanism:

\[
I = H(C)
\]

A model artifact should be identified by the bytes it contains, not by the URL, bucket, repository, server, or mirror where those bytes happen to be stored.

This notebook builds a small executable content-addressing workflow for model distribution.

## 1. Context

Location-based distribution asks:

> Where is the file hosted?

Content-addressed distribution asks:

> What is the file?

The distinction matters because locations change. Domains expire, buckets move, mirrors disappear, repository policies shift, and links break. A content address persists as long as the bytes persist.

Notebook 07 develops four claims:

1. A content hash specifies object identity.
2. Moving the object changes location but preserves identity.
3. Changing bytes changes identity.
4. Chunk manifests make large model files distributable in parts.

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import shutil
import sys
import textwrap
import zipfile
from dataclasses import dataclass, asdict

import matplotlib.pyplot as plt
import pandas as pd

# Robust paths whether run from repo root or notebooks/.
CWD = Path.cwd()
if (CWD / "src").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "src").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = Path("..").resolve()

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "07_content_addressing"
SITE = REPO_ROOT / "site"

FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
SITE.mkdir(parents=True, exist_ok=True)

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    from open_model_distribution.hashing import content_id, sha256_file
except Exception:
    def sha256_file(path, chunk_size=1024 * 1024):
        digest = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for chunk in iter(lambda: handle.read(chunk_size), b""):
                digest.update(chunk)
        return digest.hexdigest()

    def content_id(path, algorithm="sha256"):
        if algorithm != "sha256":
            raise ValueError("Only sha256 is currently supported.")
        return f"sha256:{sha256_file(path)}"

print(f"Repo root: {REPO_ROOT}")
print(f"Results:   {RESULTS}")

## 2. Content identity

The central equation is:

\[
I = H(C)
\]

where:

- \(C\) is the artifact content.
- \(H\) is a cryptographic hash function.
- \(I\) is the content identity.

The filename is metadata. The path is metadata. The host is metadata. The digest is the object identity.

In [ ]:
artifact_dir = RESULTS / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)

model_path = artifact_dir / "toy-weights.bin"

# Deterministic toy "weights" payload.
payload = b"open-model-distribution\n" + bytes(range(256)) * 128
model_path.write_bytes(payload)

digest = sha256_file(model_path)
cid = content_id(model_path)

identity_record = {
    "artifact": str(model_path.relative_to(REPO_ROOT)),
    "size_bytes": model_path.stat().st_size,
    "algorithm": "sha256",
    "digest": digest,
    "content_id": cid,
    "statement": "Identity follows content bytes, not storage location.",
}

identity_path = RESULTS / "07_content_identity.json"
identity_path.write_text(json.dumps(identity_record, indent=2), encoding="utf-8")

identity_record

## 3. Location changes, identity persists

A content-addressed object can be copied into different locations. If the bytes are identical, the identity is identical.

\[
I(C_1)=I(C_2)
\]

when \(C_1\) and \(C_2\) are the same bytes.

In [ ]:
locations = [
    RESULTS / "mirror_a" / "weights.bin",
    RESULTS / "mirror_b" / "renamed-model-object.dat",
    RESULTS / "archive" / "2026" / "toy-open-model.bin",
]

for location in locations:
    location.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(model_path, location)

location_rows = [{
    "label": "source",
    "location": str(model_path.relative_to(REPO_ROOT)),
    "size_bytes": model_path.stat().st_size,
    "content_id": content_id(model_path),
}]

for i, location in enumerate(locations, start=1):
    location_rows.append({
        "label": f"copy_{i}",
        "location": str(location.relative_to(REPO_ROOT)),
        "size_bytes": location.stat().st_size,
        "content_id": content_id(location),
    })

location_df = pd.DataFrame(location_rows)
location_df["same_identity_as_source"] = location_df["content_id"].eq(cid)

location_csv = RESULTS / "07_location_identity.csv"
location_df.to_csv(location_csv, index=False)

location_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))

labels = location_df["label"].tolist()
same = location_df["same_identity_as_source"].astype(int).tolist()

ax.bar(labels, same)
ax.set_ylim(0, 1.2)
ax.set_ylabel("same content identity")
ax.set_title("Location changes; content identity persists")
ax.set_yticks([0, 1])
ax.set_yticklabels(["different", "same"])

for idx, row in location_df.iterrows():
    short_id = row["content_id"].split(":")[1][:10] + "…"
    ax.text(idx, 1.04, short_id, ha="center", va="bottom", fontsize=9)

fig.tight_layout()
identity_location_fig = FIGURES / "07_identity_vs_location.png"
fig.savefig(identity_location_fig, dpi=180)
plt.show()

identity_location_fig

## 4. Changed bytes, changed identity

Content addressing is strict. If the bytes change, the content identity changes.

\[
C_1 \ne C_2 \Longrightarrow H(C_1) \ne H(C_2)
\]

for practical engineering purposes under a collision-resistant hash function.

In [ ]:
tampered_path = artifact_dir / "toy-weights-changed.bin"
tampered = bytearray(model_path.read_bytes())
tampered[128] = (tampered[128] + 1) % 256
tampered_path.write_bytes(bytes(tampered))

change_df = pd.DataFrame([
    {
        "artifact": "original",
        "location": str(model_path.relative_to(REPO_ROOT)),
        "content_id": content_id(model_path),
        "matches_published_identity": content_id(model_path) == cid,
    },
    {
        "artifact": "changed one byte",
        "location": str(tampered_path.relative_to(REPO_ROOT)),
        "content_id": content_id(tampered_path),
        "matches_published_identity": content_id(tampered_path) == cid,
    },
])

change_csv = RESULTS / "07_changed_bytes.csv"
change_df.to_csv(change_csv, index=False)

change_df

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))

values = change_df["matches_published_identity"].astype(int)
ax.bar(change_df["artifact"], values)
ax.set_ylim(0, 1.2)
ax.set_ylabel("verifies against published ID")
ax.set_title("Changed bytes produce a changed identity")
ax.set_yticks([0, 1])
ax.set_yticklabels(["no", "yes"])

for idx, row in change_df.iterrows():
    short_id = row["content_id"].split(":")[1][:12] + "…"
    ax.text(idx, values.iloc[idx] + 0.05, short_id, ha="center", va="bottom", fontsize=9)

fig.tight_layout()
changed_bytes_fig = FIGURES / "07_changed_bytes.png"
fig.savefig(changed_bytes_fig, dpi=180)
plt.show()

changed_bytes_fig

## 5. Content addressing pipeline

The minimal engineering pipeline is:

\[
\text{bytes}
\rightarrow
\text{hash}
\rightarrow
\text{content ID}
\rightarrow
\text{manifest}
\rightarrow
\text{verified retrieval}
\]

The manifest is the bridge between identity and distribution. It records what object should be retrieved and what digest must be verified.

In [ ]:
manifest = {
    "schema": "open-model-distribution.content-manifest.v0",
    "name": "toy-open-model",
    "artifact": str(model_path.relative_to(REPO_ROOT)),
    "size_bytes": model_path.stat().st_size,
    "algorithm": "sha256",
    "digest": digest,
    "content_id": cid,
    "locations_observed": location_df["location"].tolist(),
    "verification_rule": "sha256(received_bytes) == published_digest",
}

manifest_path = RESULTS / "07_content_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

manifest

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.axis("off")

nodes = [
    ("Model bytes", 0.08),
    ("SHA-256", 0.28),
    ("Content ID", 0.48),
    ("Manifest", 0.68),
    ("Verified retrieval", 0.88),
]

for label, x in nodes:
    ax.text(
        x, 0.55, label,
        ha="center", va="center", fontsize=14,
        bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="black", lw=1.6),
        transform=ax.transAxes,
    )

for (_, x1), (_, x2) in zip(nodes[:-1], nodes[1:]):
    ax.annotate(
        "", xy=(x2 - 0.08, 0.55), xytext=(x1 + 0.08, 0.55),
        arrowprops=dict(arrowstyle="->", lw=2),
        xycoords=ax.transAxes,
    )

ax.set_title("Content addressing pipeline", fontsize=16, pad=18)
fig.tight_layout()

pipeline_fig = FIGURES / "07_content_addressing_pipeline.png"
fig.savefig(pipeline_fig, dpi=180)
plt.show()

pipeline_fig

## 6. Preview: Chunk manifests

Large model artifacts are usually too large to move as one indivisible file. A distribution system can split a model into transfer units, hash each part, and record those chunk identities in a manifest.

Notebook 17 develops chunking as its own engineering specification. Here we only show how content identities extend to independently addressable transfer units.

In [ ]:
def chunk_file(path: Path, chunk_size: int):
    data = path.read_bytes()
    chunks = []
    for idx in range(0, len(data), chunk_size):
        chunk = data[idx:idx + chunk_size]
        chunks.append({
            "chunk_index": len(chunks),
            "offset_start": idx,
            "offset_end": idx + len(chunk),
            "size_bytes": len(chunk),
            "sha256": hashlib.sha256(chunk).hexdigest(),
        })
    return chunks

chunks = chunk_file(model_path, chunk_size=512)
chunk_df = pd.DataFrame(chunks)
chunk_csv = RESULTS / "07_chunk_table.csv"
chunk_df.to_csv(chunk_csv, index=False)

chunk_manifest = {
    "artifact_content_id": cid,
    "artifact_size_bytes": model_path.stat().st_size,
    "chunk_size_bytes": 512,
    "chunk_count": len(chunks),
    "chunks": chunks,
}

chunk_manifest_path = RESULTS / "07_chunk_manifest.json"
chunk_manifest_path.write_text(json.dumps(chunk_manifest, indent=2), encoding="utf-8")

chunk_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.axis("off")

# Conceptual chunking diagram rather than a dense debug chart.
ax.text(0.08, 0.72, "model.bin", ha="center", va="center", fontsize=14,
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black", lw=1.5),
        transform=ax.transAxes)

# Arrow from model to chunks
ax.annotate("", xy=(0.20, 0.72), xytext=(0.13, 0.72),
            arrowprops=dict(arrowstyle="->", lw=2), xycoords=ax.transAxes)

# Chunk blocks
chunk_x0 = 0.23
chunk_w = 0.08
for i in range(5):
    label = f"chunk {i}" if i < 4 else "..."
    x = chunk_x0 + i * (chunk_w + 0.02)
    ax.add_patch(plt.Rectangle((x, 0.60), chunk_w, 0.24,
                               fill=False, lw=1.5, transform=ax.transAxes))
    ax.text(x + chunk_w / 2, 0.72, label, ha="center", va="center",
            fontsize=10, transform=ax.transAxes)

ax.text(0.43, 0.50, "512-byte transfer units", ha="center", va="center",
        fontsize=11, transform=ax.transAxes)

# Arrow to hashing
ax.annotate("", xy=(0.63, 0.72), xytext=(0.55, 0.72),
            arrowprops=dict(arrowstyle="->", lw=2), xycoords=ax.transAxes)

ax.text(0.70, 0.72, "hash each chunk", ha="center", va="center", fontsize=13,
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black", lw=1.5),
        transform=ax.transAxes)

ax.annotate("", xy=(0.83, 0.72), xytext=(0.77, 0.72),
            arrowprops=dict(arrowstyle="->", lw=2), xycoords=ax.transAxes)

ax.text(0.91, 0.72, "chunk\nmanifest", ha="center", va="center", fontsize=13,
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black", lw=1.5),
        transform=ax.transAxes)

# Hash examples from the generated chunk table, but keep them readable.
preview_hashes = chunk_df.head(3)["sha256"].str[:10].tolist()
for j, short in enumerate(preview_hashes):
    ax.text(0.70, 0.46 - j * 0.09, f"h{j} = {short}…", ha="center",
            va="center", fontsize=10, transform=ax.transAxes)

ax.set_title("Preview: chunk manifests extend content identity", fontsize=16, pad=16)
fig.tight_layout()

chunk_fig = FIGURES / "07_chunk_manifest.png"
fig.savefig(chunk_fig, dpi=180)
plt.show()

chunk_fig

## 7. Location addressing versus content addressing

The design transition is:

\[
\text{location} \rightarrow \text{content identity}
\]

Location addressing depends on an administrative path. Content addressing depends on a digest of the bytes.

In [ ]:
comparison = pd.DataFrame([
    {
        "distribution_style": "location-based",
        "identifier": "URL / repository path / bucket key",
        "changes_when_location_moves": True,
        "verifies_bytes_directly": False,
        "engineering_risk": "link rot, takedown, mirror drift",
    },
    {
        "distribution_style": "content-addressed",
        "identifier": "cryptographic digest",
        "changes_when_location_moves": False,
        "verifies_bytes_directly": True,
        "engineering_risk": "requires manifest discipline and hash verification",
    },
])

comparison_path = RESULTS / "07_addressing_comparison.csv"
comparison.to_csv(comparison_path, index=False)

comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.axis("off")

left = [("URL", 0.18, 0.75), ("Server", 0.18, 0.50), ("File", 0.18, 0.25)]
right = [("Bytes", 0.72, 0.75), ("Hash", 0.72, 0.50), ("Identity", 0.72, 0.25)]

ax.text(0.18, 0.95, "Location addressing", ha="center", va="center", fontsize=15, weight="bold", transform=ax.transAxes)
ax.text(0.72, 0.95, "Content addressing", ha="center", va="center", fontsize=15, weight="bold", transform=ax.transAxes)

for label, x, y in left + right:
    ax.text(x, y, label, ha="center", va="center", fontsize=14,
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black", lw=1.4),
            transform=ax.transAxes)

for nodes in [left, right]:
    for (_, x1, y1), (_, x2, y2) in zip(nodes[:-1], nodes[1:]):
        ax.annotate("", xy=(x2, y2 + 0.08), xytext=(x1, y1 - 0.08),
                    arrowprops=dict(arrowstyle="->", lw=2), xycoords=ax.transAxes)

ax.text(0.18, 0.08, "Where is it?", ha="center", fontsize=12, transform=ax.transAxes)
ax.text(0.72, 0.08, "What is it?", ha="center", fontsize=12, transform=ax.transAxes)

fig.tight_layout()
comparison_fig = FIGURES / "07_location_vs_content_addressing.png"
fig.savefig(comparison_fig, dpi=180)
plt.show()

comparison_fig

## 8. Engineering statement

Content addressing specifies model distribution by deriving object identity from the artifact itself. When the identifier is the digest of the bytes, model distribution becomes location-independent, tamper-evident, and reproducible across mirrors, repositories, and peer-to-peer networks.

## 9. Summary table

| Property | Specification |
|---|---|
| Identity | Content addressing |
| Verification | SHA-256 digest comparison |
| Distribution | Content manifest |
| Preview | Chunk manifest |

Notebook 07 stays centered on identity. Chunking appears only as a preview because notebook 17 develops transfer units as a separate engineering specification.

## 10. Key equations

Content identity:

\[
I = H(C)
\]

Identity preservation, assuming a collision-resistant hash function:

\[
I(C_1)=I(C_2) \iff C_1=C_2
\]

Verification condition:

\[
V=\bigl(H(C_{\mathrm{received}})=H(C_{\mathrm{published}})\bigr)
\]

Identity differs from location:

\[
I \ne \text{location}
\]

## 11. Notebook outputs

Notebook 07 writes:

- `results/07_content_addressing/07_content_identity.json`
- `results/07_content_addressing/07_content_manifest.json`
- `results/07_content_addressing/07_chunk_manifest.json`
- `results/07_content_addressing/07_location_identity.csv`
- `results/07_content_addressing/07_changed_bytes.csv`
- `results/07_content_addressing/07_chunk_table.csv`
- `results/07_content_addressing/07_addressing_comparison.csv`
- `figures/07_identity_vs_location.png`
- `figures/07_changed_bytes.png`
- `figures/07_content_addressing_pipeline.png`
- `figures/07_chunk_manifest.png`
- `figures/07_location_vs_content_addressing.png`

In [ ]:
notebook_manifest = {
    "notebook": "07_content_addressing.ipynb",
    "title": "Content Addressing Specifies Model Distribution",
    "primary_specification": "content addressing",
    "statement": "Identity follows content, not storage location.",
    "equations": [
        "I = H(C)",
        "I(C_1)=I(C_2) iff C_1=C_2, assuming collision resistance",
        "V = (H(C_received)=H(C_published))",
        "I != location",
        "Summary table maps identity, verification, distribution, and preview specifications",
    ],
    "outputs": {
        "content_identity": str(identity_path.relative_to(REPO_ROOT)),
        "content_manifest": str(manifest_path.relative_to(REPO_ROOT)),
        "chunk_manifest": str(chunk_manifest_path.relative_to(REPO_ROOT)),
        "location_identity_csv": str(location_csv.relative_to(REPO_ROOT)),
        "changed_bytes_csv": str(change_csv.relative_to(REPO_ROOT)),
        "chunk_table_csv": str(chunk_csv.relative_to(REPO_ROOT)),
        "addressing_comparison_csv": str(comparison_path.relative_to(REPO_ROOT)),
        "identity_vs_location_figure": str(identity_location_fig.relative_to(REPO_ROOT)),
        "changed_bytes_figure": str(changed_bytes_fig.relative_to(REPO_ROOT)),
        "pipeline_figure": str(pipeline_fig.relative_to(REPO_ROOT)),
        "chunk_manifest_figure": str(chunk_fig.relative_to(REPO_ROOT)),
        "comparison_figure": str(comparison_fig.relative_to(REPO_ROOT)),
    },
}

notebook_manifest_path = RESULTS / "07_notebook_manifest.json"
notebook_manifest_path.write_text(json.dumps(notebook_manifest, indent=2), encoding="utf-8")

notebook_manifest

## 12. Download artifacts

Run the final cell to package notebook 07 outputs for download.

In [ ]:
# Package notebook artifacts for download

from pathlib import Path
from IPython.display import FileLink, display
import zipfile

NOTEBOOKS = REPO_ROOT / "notebooks"
SITE = REPO_ROOT / "site"

zip_path = RESULTS / "07_content_addressing_artifacts.zip"

notebook_path = NOTEBOOKS / "07_content_addressing.ipynb"
fallback_notebook_path = Path.cwd() / "07_content_addressing.ipynb"

paths_to_include = [
    notebook_path if notebook_path.exists() else fallback_notebook_path,
    FIGURES,
    RESULTS,
    SITE,
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        item = Path(item)

        if item.is_dir():
            for path in item.rglob("*"):
                if path.is_file() and path.resolve() != zip_path.resolve():
                    try:
                        archive_name = path.relative_to(REPO_ROOT)
                    except ValueError:
                        archive_name = path.name
                    zf.write(path, archive_name)

        elif item.exists() and item.resolve() != zip_path.resolve():
            try:
                archive_name = item.relative_to(REPO_ROOT)
            except ValueError:
                archive_name = item.name
            zf.write(item, archive_name)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")

display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass